# Ablation Study — Training and Evaluation

Notebook này triển khai quy trình huấn luyện và đánh giá các mô hình trong thực nghiệm Ablation Study của DeepKoopFormer.

Quy trình bao gồm:
- Nạp dữ liệu đã tiền xử lý.
- Nạp các định nghĩa mô hình từ `02_ablation_model.ipynb`.
- Xây dựng sliding windows cho Train/Test.
- Tính prediction loss và Lyapunov regularization loss.
- Huấn luyện bằng Adam optimizer.
- Đánh giá bằng MSE và MAE.
- Lưu checkpoint, phục hồi quá trình huấn luyện và quản lý các cấu hình Ablation.

## 1. Training Environment and Configuration

Thiết lập môi trường PyTorch, seed ngẫu nhiên và cấu hình thực nghiệm.

Bốn dataset được nạp trực tiếp từ `01_ablation_data.ipynb` thông qua các file `.pt` đã chuẩn bị.

Các định nghĩa mô hình được trích xuất từ `02_ablation_model.ipynb` để bảo đảm notebook Training sử dụng cùng kiến trúc với notebook Model.

Phần này không thực hiện huấn luyện hoặc tạo lại dữ liệu.

In [1]:
from pathlib import Path
import ast
import json
import random
import hashlib
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# 1. Reproducibility
SEED = 7

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Locate project root
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "05_ablation"
            / "data" / "ablation_manifest.csv"
        ).is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate ablation project."
    )

EXPERIMENT_DIR = (
    PROJECT_ROOT / "experiments" / "05_ablation"
)

DATA_DIR = EXPERIMENT_DIR / "data"
RESULTS_DIR = EXPERIMENT_DIR / "results"

MODEL_NOTEBOOK = (
    EXPERIMENT_DIR / "02_ablation_model.ipynb"
)

assert MODEL_NOTEBOOK.is_file()

# 3. Load four validated datasets
DATASET_FILES = {
    "Electricity": "electricity_ablation.pt",
    "Financial": "financial_ablation.pt",
    "CMIP6 Wind": "cmip6_wind_ablation.pt",
    "CMIP6 Pressure": "cmip6_pressure_candidate_2500.pt"
}

ABLATION_DATASETS = {}

for name, filename in DATASET_FILES.items():

    path = DATA_DIR / filename

    bundle = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    assert bundle["train"].ndim == 2
    assert bundle["test"].ndim == 2

    assert (
        bundle["train"].shape[1]
        == bundle["test"].shape[1]
        == bundle["input_dim"]
    )

    assert torch.isfinite(bundle["train"]).all()
    assert torch.isfinite(bundle["test"]).all()

    ABLATION_DATASETS[name] = bundle

# 4. Author's ablation configuration
PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

MODEL_VARIANTS = [
    "DLinear",
    "LSTM",
    "PatchTST",
    "PatchTST+LinearLatent",
    "KoopPatchTST_NoLyap",
    "KoopPatchTST_Full"
]

D_MODEL = 96
D_LATENT = 96

LSTM_HIDDEN_DIM = 96
LSTM_NUM_LAYERS = 2

NUM_ENCODER_LAYERS = 3
NUM_ATTENTION_HEADS = 4
DIM_FEEDFORWARD = 96

RHO_MAX = 0.99
LYAP_WEIGHT = 0.1

EPOCHS = 4000
LEARNING_RATE = 3e-4

# 5. Extract model definitions from File 2
REQUIRED_DEFINITIONS = {
    "_orth",
    "StrictStableKoopmanOperator",
    "LinearLatentPropagator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "PatchTSTBaseline",
    "PatchTST_UnconstrainedLatent",
    "Koopformer_PatchTST",
    "SimpleLSTMForecaster",
    "DLinearForecaster",
    "create_ablation_model"
}

with open(
    MODEL_NOTEBOOK, "r", encoding="utf-8"
) as f:
    model_notebook = json.load(f)

definitions = []
found_names = set()

for cell in model_notebook["cells"]:

    if cell["cell_type"] != "code":
        continue

    source = cell["source"]

    if isinstance(source, list):
        source = "".join(source)

    if not source.strip():
        continue

    tree = ast.parse(source)

    for node in tree.body:

        if isinstance(
            node, (ast.ClassDef, ast.FunctionDef)
        ) and node.name in REQUIRED_DEFINITIONS:

            definitions.append(node)
            found_names.add(node.name)

# 6. Ensure no model definition is missing
missing = REQUIRED_DEFINITIONS - found_names

if missing:
    raise RuntimeError(
        f"Missing model definitions: {sorted(missing)}"
    )

# 7. Execute only the selected definitions
module = ast.Module(
    body=definitions,
    type_ignores=[]
)

ast.fix_missing_locations(module)

exec(
    compile(
        module,
        str(MODEL_NOTEBOOK),
        "exec"
    ),
    globals()
)

c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()



## 2. Train/Test Sliding Window Preparation

Phần này chuyển các chuỗi thời gian đã tiền xử lý thành các mẫu đầu vào và nhãn dự báo bằng kỹ thuật sliding window.

### build_ablation_windows()

Với dữ liệu đầu vào có kích thước `(N, F)`, hàm tạo:

- `X`: `(samples, patch_len, F)`
- `Y`: `(samples, F × horizon)`

Mỗi input window chứa `patch_len` bước thời gian quá khứ. Target chứa các giá trị ở `horizon` bước thời gian tiếp theo, được sắp xếp theo từng channel.

### prepare_ablation_data()

Sử dụng hàm sliding window để tạo bốn tensor `x_train`, `y_train`, `x_test`, `y_test`.

Train và Test được xử lý độc lập, không tạo các window đi qua ranh giới giữa hai tập.

Dữ liệu đầu vào đã được chuẩn hóa trong notebook Data nên không thực hiện scaling lại.

Giữ nguyên quy ước về số lượng window và cách sắp xếp target đã sử dụng trong thực nghiệm.

In [2]:
# 1. Sliding-window function (author's implementation)
def build_ablation_windows(
    data,
    patch_len: int,
    horizon: int
):
    """
    Input:
        data: (N, F)
        patch_len: number of past timestamps
        horizon: number of future timestamps

    Returns:
        X: (samples, patch_len, F)
        Y: (samples, F * horizon)

    Matches the author's sliding-window convention.
    """

    if isinstance(data, torch.Tensor):
        data = data.detach().cpu().numpy()

    data = np.asarray(data, dtype=np.float32)

    if data.ndim != 2:
        raise ValueError("Expected data shape (N, F).")

    n_windows = len(data) - patch_len - horizon

    if n_windows <= 0:
        raise ValueError(
            f"Insufficient data: N={len(data)}, "
            f"P={patch_len}, H={horizon}"
        )

    X, Y = [], []

    for i in range(n_windows):

        X.append(
            data[i:i + patch_len]
        )

        Y.append(
            data[
                i + patch_len:
                i + patch_len + horizon
            ].T.reshape(-1)
        )

    x_in = torch.tensor(
        np.stack(X),
        dtype=torch.float32
    )

    x_out = torch.tensor(
        np.stack(Y),
        dtype=torch.float32
    )

    return x_in, x_out


# 2. Unified dataset preparation function
def prepare_ablation_data(
    dataset_name: str,
    patch_len: int,
    horizon: int
):
    """
    Prepare Train/Test tensors for one configuration.

    Data are already scaled in File 1.
    Keep tensors on CPU until actual training.
    """

    if dataset_name not in ABLATION_DATASETS:
        raise ValueError(
            f"Unknown dataset: {dataset_name}"
        )

    if patch_len not in PATCH_LENGTHS:
        raise ValueError(
            f"Unsupported patch length: {patch_len}"
        )

    if horizon not in HORIZONS:
        raise ValueError(
            f"Unsupported horizon: {horizon}"
        )

    bundle = ABLATION_DATASETS[dataset_name]

    train = bundle["train"]
    test = bundle["test"]

    # Create windows separately to preserve split boundary
    x_train, y_train = build_ablation_windows(
        train,
        patch_len,
        horizon
    )

    x_test, y_test = build_ablation_windows(
        test,
        patch_len,
        horizon
    )

    # Validate shapes
    F = bundle["input_dim"]

    assert x_train.shape == (
        len(train) - patch_len - horizon,
        patch_len,
        F
    )

    assert y_train.shape == (
        len(x_train),
        F * horizon
    )

    assert x_test.shape == (
        len(test) - patch_len - horizon,
        patch_len,
        F
    )

    assert y_test.shape == (
        len(x_test),
        F * horizon
    )

    # Validate numerical values
    for tensor in [
        x_train, y_train,
        x_test, y_test
    ]:
        assert torch.isfinite(tensor).all()

    return x_train, y_train, x_test, y_test

## 3. Prediction Loss and Lyapunov Regularization

Phần này định nghĩa hàm `compute_ablation_loss()` để tính loss dùng trong quá trình huấn luyện.

Prediction loss sử dụng Mean Squared Error (MSE) giữa giá trị dự báo và ground truth.

Đối với mô hình Full, hàm bổ sung thành phần chính quy hóa Lyapunov nhằm phạt sự gia tăng năng lượng của latent representation sau phép biến đổi Koopman.

Tổng loss:

`Total Loss = Prediction Loss + λ × Lyapunov Loss`

Trong đó:
- `λ = 0` với các mô hình không sử dụng chính quy hóa Lyapunov.
- `λ = 0.1` với KoopPatchTST_Full.

Hàm trả về tổng loss, prediction loss, Lyapunov loss và prediction tensor.

Công thức tính loss được giữ nguyên theo implementation đã sử dụng trong thực nghiệm.

In [3]:
# 1. Loss function following the author's implementation
def compute_ablation_loss(
    model,
    x,
    y,
    lyap_weight=0.0
):
    """
    Returns:
        total_loss
        prediction_loss
        lyapunov_loss
        predictions
    """

    # Forward pass
    pred, z, z_next = model(
        x,
        return_latents=True
    )

    # Prediction loss
    pred_loss = F.mse_loss(pred, y)

    # Lyapunov regularization
    if lyap_weight:

        P = torch.eye(
            z.shape[1],
            device=z.device,
            dtype=z.dtype
        )

        zp = torch.einsum(
            "bi,ij->bj", z, P
        )

        znp = torch.einsum(
            "bi,ij->bj", z_next, P
        )

        lyap_loss = torch.relu(
            (znp * z_next).sum(1)
            - (zp * z).sum(1)
        ).mean()

    else:
        lyap_loss = torch.zeros(
            (), device=x.device, dtype=x.dtype
        )

    total_loss = (
        pred_loss + lyap_weight * lyap_loss
    )

    return total_loss, pred_loss, lyap_loss, pred


## 4. Unified Ablation Training Loop

Hàm `train_ablation_model()` triển khai quy trình huấn luyện thống nhất cho sáu mô hình Ablation.

Quy trình huấn luyện:
1. Chuyển mô hình và dữ liệu lên thiết bị tính toán.
2. Khởi tạo hoặc phục hồi Adam optimizer.
3. Thực hiện forward pass và tính tổng loss.
4. Lan truyền ngược để tính gradient.
5. Cập nhật các tham số mô hình.
6. Ghi lại training history.
7. Lưu checkpoint theo chu kỳ nếu được cấu hình.

### Training configuration

- Optimizer: Adam
- Learning rate: 3e-4
- Epochs: 4000
- Training mode: Full-batch
- Seed: 7

Training history bao gồm:
- `loss`: Tổng loss.
- `mse`: Prediction loss.
- `lyap_weighted`: Lyapunov loss sau khi nhân hệ số chính quy hóa.
- `spectral_proxy`: Giá trị theo dõi liên quan đến toán tử Koopman.

Hàm hỗ trợ tiếp tục huấn luyện từ checkpoint thông qua optimizer, training history và epoch đã lưu.

Không thay đổi quy trình cập nhật trọng số trong lần refactor này.

In [5]:
def train_ablation_model(
    model,
    x_train,
    y_train,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    koop_attr=None,
    lyap_weight=0.0,
    device=DEVICE,
    optimizer=None,
    start_epoch=0,
    history=None,
    checkpoint_callback=None,
    checkpoint_interval=500,
    verbose=True
):
    """
    Full-batch training following the author's implementation.

    Optional arguments support checkpoint/resume
    without changing the optimization procedure.
    """

    # 1. Validate inputs
    assert x_train.ndim == 3
    assert y_train.ndim == 2
    assert len(x_train) == len(y_train)
    assert 0 <= start_epoch < epochs

    if start_epoch > 0 and optimizer is None:
        raise ValueError(
            "Resuming requires a restored optimizer."
        )

    model = model.to(device)

    x_train = x_train.to(device)
    y_train = y_train.to(device)

    # 2. Adam optimizer
    if optimizer is None:
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr
        )

    # 3. Training history
    history_keys = [
        "loss",
        "mse",
        "lyap_weighted",
        "spectral_proxy"
    ]

    if history is None:
        history = {
            key: [] for key in history_keys
        }

    for key in history_keys:
        assert key in history
        assert len(history[key]) == start_epoch

    # 4. Full-batch training
    for ep in range(start_epoch, epochs):

        model.train()
        optimizer.zero_grad()

        # Forward + loss from Cell 3
        total_loss, pred_loss, lyap_loss, pred = (
            compute_ablation_loss(
                model,
                x_train,
                y_train,
                lyap_weight=lyap_weight
            )
        )

        if not torch.isfinite(total_loss).item():
            raise FloatingPointError(
                f"Non-finite loss at epoch {ep}"
            )

        # Backpropagation
        total_loss.backward()

        # Update parameters
        optimizer.step()

        # 5. Record losses
        history["loss"].append(
            float(total_loss.detach().item())
        )

        history["mse"].append(
            float(pred_loss.detach().item())
        )

        history["lyap_weighted"].append(
            float(
                (lyap_weight * lyap_loss)
                .detach().item()
            )
        )

        # 6. Log the same spectral proxy as the author
        koop = (
            getattr(model, koop_attr, None)
            if koop_attr is not None
            else None
        )

        if isinstance(
            koop, StrictStableKoopmanOperator
        ):
            spectral_proxy = (
                torch.sigmoid(koop.S_raw)
                .detach()
                .abs()
                .max()
                .item()
            )
        else:
            spectral_proxy = 0.0

        history["spectral_proxy"].append(
            float(spectral_proxy)
        )

        # 7. Progress logging
        if verbose and (
            ep % max(1, epochs // 10) == 0
            or ep == epochs - 1
        ):
            print(
                f"Epoch {ep:>4}/{epochs} | "
                f"MSE={pred_loss.item():.6e} | "
                f"Lyap={(lyap_weight * lyap_loss).item():.6e} | "
                f"Total={total_loss.item():.6e}"
            )

        # 8. Optional checkpoint callback
        if (
            checkpoint_callback is not None
            and (
                (ep + 1) % checkpoint_interval == 0
                or ep == epochs - 1
            )
        ):
            checkpoint_callback(
                ep + 1,
                model,
                optimizer,
                history
            )

    # 9. Final Train predictions
    model.eval()

    with torch.no_grad():
        pred_train = (
            model(x_train)
            .detach()
            .cpu()
            .numpy()
        )

    # 10. Return all training information
    return {
        "model": model,
        "optimizer": optimizer,
        "pred_train": pred_train,
        "history": history,
        "epochs_completed": epochs
    }

## 5. Train/Test Evaluation

Phần này triển khai quy trình đánh giá mô hình sau khi huấn luyện.

### calculate_regression_metrics()

Tính hai chỉ số đánh giá:
- MSE: Mean Squared Error.
- MAE: Mean Absolute Error.

### predict_ablation_model()

Thực hiện inference theo các batch để giảm mức sử dụng bộ nhớ.

Mô hình được chuyển sang evaluation mode và sử dụng `torch.no_grad()` nhằm tránh tính gradient.

### evaluate_ablation_model()

Đánh giá mô hình trên cả Train và Test, trả về:
- MSE và MAE.
- Số lượng mẫu.
- Predicted values.
- Ground-truth values.

Quá trình evaluation không gọi optimizer và không cập nhật trọng số mô hình.

Các kết quả được sử dụng để tổng hợp hiệu năng của sáu biến thể Ablation.

In [6]:
# 1. Regression metrics
def calculate_regression_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    assert y_true.shape == y_pred.shape
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    errors = y_pred - y_true

    mse = float(np.mean(errors ** 2))
    mae = float(np.mean(np.abs(errors)))

    return {
        "MSE": mse,
        "MAE": mae
    }


# 2. Batched prediction (inference only)
@torch.no_grad()
def predict_ablation_model(
    model,
    x,
    device=DEVICE,
    batch_size=128
):
    if batch_size <= 0:
        raise ValueError("batch_size must be positive.")

    if len(x) == 0:
        raise ValueError("Cannot predict an empty dataset.")

    model = model.to(device)

    was_training = model.training
    model.eval()

    predictions = []

    try:
        for start in range(0, len(x), batch_size):

            x_batch = x[
                start:start + batch_size
            ].to(device)

            pred_batch = model(x_batch)

            predictions.append(
                pred_batch.detach().cpu().numpy()
            )

    finally:
        model.train(was_training)

    return np.concatenate(
        predictions, axis=0
    )


# 3. Unified Train/Test evaluation
def evaluate_ablation_model(
    model,
    x_train,
    y_train,
    x_test,
    y_test,
    device=DEVICE,
    batch_size=128
):
    """
    Evaluate one trained model on Train and Test.

    Returns metrics, predictions and ground truth.
    Does not update model parameters.
    """

    results = {}

    for split_name, x, y in [
        ("Train", x_train, y_train),
        ("Test", x_test, y_test)
    ]:

        pred = predict_ablation_model(
            model,
            x,
            device=device,
            batch_size=batch_size
        )

        truth = y.detach().cpu().numpy()

        assert pred.shape == truth.shape
        assert pred.shape[0] == len(x)

        metrics = calculate_regression_metrics(
            truth,
            pred
        )

        results[split_name] = {
            "MSE": metrics["MSE"],
            "MAE": metrics["MAE"],
            "Samples": len(x),
            "predictions": pred,
            "ground_truth": truth
        }

    return results

## 6. Checkpoint and Resume Training

Phần này triển khai cơ chế lưu và phục hồi trạng thái huấn luyện, cho phép tiếp tục thực nghiệm nếu quá trình training bị gián đoạn.

### to_cpu_copy()

Tạo bản sao của các tensor trên CPU để phục vụ lưu checkpoint.

### save_ablation_checkpoint()

Lưu các thành phần cần thiết cho việc phục hồi:
- Model state dictionary.
- Optimizer state dictionary.
- Training history.
- Epoch hiện tại.
- Experiment configuration.
- Random number generator states.

Checkpoint được ghi thông qua file tạm trước khi thay thế file đích nhằm giảm rủi ro lưu file không hoàn chỉnh.

### load_ablation_checkpoint()

Phục hồi kiến trúc mô hình, trọng số, optimizer, training history và trạng thái ngẫu nhiên từ checkpoint đã lưu.

Cấu hình của checkpoint được đối chiếu với cấu hình thực nghiệm trước khi phục hồi.

Cơ chế này phục vụ việc tiếp tục huấn luyện từ trạng thái trước đó thay vì bắt đầu lại từ đầu.

In [7]:
import copy

# 1. Convert nested tensors to CPU for portable checkpoints
def to_cpu_copy(obj):

    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().clone()

    if isinstance(obj, dict):
        return {
            k: to_cpu_copy(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [to_cpu_copy(v) for v in obj]

    if isinstance(obj, tuple):
        return tuple(to_cpu_copy(v) for v in obj)

    return copy.deepcopy(obj)


# 2. Save full training state
def save_ablation_checkpoint(
    epoch, model, optimizer, history,
    path, config
):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # Store NumPy RNG in primitive Python types
    np_state = np.random.get_state()

    checkpoint = {
        "version": 1,
        "epoch": int(epoch),
        "config": dict(config),

        "model_state_dict": to_cpu_copy(
            model.state_dict()
        ),

        "optimizer_state_dict": to_cpu_copy(
            optimizer.state_dict()
        ),

        "history": copy.deepcopy(history),

        "rng_torch": torch.get_rng_state(),
        "rng_cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),

        "rng_numpy": {
            "name": np_state[0],
            "keys": np_state[1].tolist(),
            "pos": int(np_state[2]),
            "has_gauss": int(np_state[3]),
            "cached_gaussian": float(np_state[4])
        },

        "rng_python": random.getstate()
    }

    # Atomic replacement: prevent partial checkpoint files
    temp_path = path.with_name(
        path.name + ".tmp"
    )

    torch.save(checkpoint, temp_path)
    temp_path.replace(path)


# 3. Restore training state
def load_ablation_checkpoint(
    path,
    expected_config,
    device=DEVICE
):

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    if checkpoint["config"] != expected_config:
        raise ValueError(
            "Checkpoint configuration does not match "
            "the requested experiment."
        )

    config = checkpoint["config"]

    # Recreate the correct model architecture
    model, koop_attr, lyap_weight = (
        create_ablation_model(
            model_name=config["model_name"],
            input_dim=config["input_dim"],
            patch_len=config["patch_len"],
            horizon=config["horizon"]
        )
    )

    model = model.to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    # Restore Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # Restore history and epoch
    history = checkpoint["history"]
    start_epoch = checkpoint["epoch"]

    assert all(
        len(values) == start_epoch
        for values in history.values()
    )

    # Restore RNG AFTER creating the model and optimizer
    torch.set_rng_state(
        checkpoint["rng_torch"]
    )

    if device.type == "cuda":
        torch.cuda.set_rng_state_all(
            checkpoint["rng_cuda"]
        )

    saved_np = checkpoint["rng_numpy"]

    np.random.set_state((
        saved_np["name"],
        np.asarray(
            saved_np["keys"], dtype=np.uint32
        ),
        saved_np["pos"],
        saved_np["has_gauss"],
        saved_np["cached_gaussian"]
    ))

    random.setstate(
        checkpoint["rng_python"]
    )

    return (
        model,
        optimizer,
        history,
        start_epoch,
        koop_attr,
        lyap_weight
    )


## 7. End-to-End Ablation Experiment Runner

Hàm `run_one_ablation()` quản lý toàn bộ quá trình thực hiện một cấu hình Ablation từ đầu đến cuối.

Các bước chính:
1. Kiểm tra dataset, model và cấu hình.
2. Xác định thư mục lưu kết quả.
3. Kiểm tra kết quả đã hoàn thành hoặc checkpoint có sẵn.
4. Khởi tạo hoặc phục hồi mô hình.
5. Huấn luyện với cấu hình được chỉ định.
6. Đánh giá trên Train và Test.
7. Lưu training history, metrics và checkpoint.
8. Giải phóng tài nguyên sau khi hoàn thành.

### Experiment artifacts

Mỗi cấu hình có thể tạo:
- `latest.pt`: Checkpoint phục vụ lưu và phục hồi.
- `metrics.json`: Các chỉ số đánh giá và metadata.
- `training_history.csv`: Lịch sử huấn luyện.
- `predictions.npz`: Predictions và ground truth, nếu được bật.

### Resume and reproducibility

Runner đối chiếu cấu hình cùng fingerprint dữ liệu nhằm phát hiện sự không tương thích giữa kết quả đã lưu và yêu cầu thực nghiệm.

Khi phát hiện kết quả hoàn chỉnh, runner có thể bỏ qua cấu hình đã thực hiện.

Không sử dụng `force_restart=True` khi muốn bảo toàn các kết quả và checkpoint hiện có.

In [8]:
import hashlib
import json
import re
import time
from pathlib import Path


def _safe_slug(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def _processed_data_hash(bundle):
    """Fingerprint the actual scaled train/test data, not only the raw file."""
    digest = hashlib.sha256()
    for split in ("train", "test"):
        arr = bundle[split].detach().cpu().contiguous().numpy()
        digest.update(split.encode("utf-8"))
        digest.update(str(arr.shape).encode("utf-8"))
        digest.update(arr.tobytes())
    return digest.hexdigest()


def run_one_ablation(
    dataset_name,
    model_name,
    patch_len,
    horizon,
    epochs=EPOCHS,
    seed=SEED,
    checkpoint_every=500,
    save_predictions=False,
    verbose=False,
    force_restart=False,
):
    """
    Train, evaluate, persist, and (if interrupted) resume one configuration.
    A completed metrics.json is the source of truth for skip-on-rerun.
    """
    if dataset_name not in ABLATION_DATASETS:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    if model_name not in MODEL_VARIANTS:
        raise ValueError(f"Unknown model: {model_name}")
    if patch_len not in PATCH_LENGTHS or horizon not in HORIZONS:
        raise ValueError(f"Invalid patch/horizon: {patch_len}, {horizon}")
    if epochs <= 0 or checkpoint_every <= 0:
        raise ValueError("epochs and checkpoint_every must be positive")

    bundle = ABLATION_DATASETS[dataset_name]
    run_dir = (
        RESULTS_DIR / "ablation_runs" / _safe_slug(dataset_name)
        / f"p{patch_len:03d}_h{horizon:02d}"
        / _safe_slug(model_name) / f"epochs{epochs}_seed{seed}"
    )
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "latest.pt"
    metrics_path = run_dir / "metrics.json"
    history_path = run_dir / "training_history.csv"
    prediction_path = run_dir / "predictions.npz"

    # The Lyapunov coefficient is tied to the variant, not user-provided.
    expected_lyap_weight = (
        float(LYAP_WEIGHT) if model_name == "KoopPatchTST_Full" else 0.0
    )
    config = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "input_dim": int(bundle["input_dim"]),
        "patch_len": int(patch_len),
        "horizon": int(horizon),
        "epochs": int(epochs),
        "lr": float(LEARNING_RATE),
        "lyap_weight": expected_lyap_weight,
        "seed": int(seed),
        "raw_sha256": str(bundle["raw_sha256"]),
        "processed_sha256": _processed_data_hash(bundle),
        "d_model": int(D_MODEL),
        "lstm_hidden_dim": int(LSTM_HIDDEN_DIM),
        "lstm_num_layers": int(LSTM_NUM_LAYERS),
        "rho_max": float(RHO_MAX),
    }

    if force_restart:
        for file_path in (checkpoint_path, metrics_path, history_path, prediction_path):
            file_path.unlink(missing_ok=True)

    if metrics_path.exists():
        with metrics_path.open("r", encoding="utf-8") as stream:
            previous = json.load(stream)
        if previous["config"] != config or previous.get("status") != "complete":
            raise ValueError(f"Existing metrics are incompatible: {metrics_path}")
        if not checkpoint_path.exists():
            raise FileNotFoundError(
                f"Metrics exist but final checkpoint is missing: {checkpoint_path}"
            )
        print(f"[SKIP — completed] {dataset_name} | {model_name} | P={patch_len}, H={horizon}")
        return previous

    # Do not leak test observations into training.
    x_train, y_train, x_test, y_test = prepare_ablation_data(
        dataset_name, patch_len, horizon
    )

    # Initializing then restoring in this order is intentional: loader restores RNG.
    set_seed(seed)
    if checkpoint_path.exists():
        model, optimizer, history, start_epoch, koop_attr, lyap_weight = (
            load_ablation_checkpoint(checkpoint_path, expected_config=config, device=DEVICE)
        )
        print(f"[RESUME] {model_name}: epoch {start_epoch}/{epochs}")
    else:
        model, koop_attr, lyap_weight = create_ablation_model(
            model_name, int(bundle["input_dim"]), patch_len, horizon
        )
        optimizer, history, start_epoch = None, None, 0
        print(f"[START]  {dataset_name} | {model_name} | P={patch_len}, H={horizon}")

    assert lyap_weight == expected_lyap_weight
    if not (0 <= start_epoch <= epochs):
        raise ValueError(f"Invalid checkpoint epoch: {start_epoch}/{epochs}")

    def save_callback(epoch, current_model, current_optimizer, current_history):
        save_ablation_checkpoint(
            epoch=epoch,
            model=current_model,
            optimizer=current_optimizer,
            history=current_history,
            path=checkpoint_path,
            config=config,
        )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    if start_epoch < epochs:
        trained = train_ablation_model(
            model=model,
            x_train=x_train,
            y_train=y_train,
            epochs=epochs,
            lr=LEARNING_RATE,
            koop_attr=koop_attr,
            lyap_weight=lyap_weight,
            device=DEVICE,
            optimizer=optimizer,
            start_epoch=start_epoch,
            history=history,
            checkpoint_callback=save_callback,
            checkpoint_interval=checkpoint_every,
            verbose=verbose,
        )
        model = trained["model"]
        optimizer = trained["optimizer"]
        history = trained["history"]
        del trained
    else:
        model.eval()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    training_seconds_this_session = time.perf_counter() - start_time

    eval_start = time.perf_counter()
    evaluation = evaluate_ablation_model(
        model, x_train, y_train, x_test, y_test,
        device=DEVICE, batch_size=128
    )
    evaluation_seconds = time.perf_counter() - eval_start

    history_df = pd.DataFrame(history)
    if len(history_df) != epochs:
        raise RuntimeError(f"Unexpected history length: {len(history_df)} != {epochs}")
    history_df.insert(0, "epoch", np.arange(1, epochs + 1))
    history_df.to_csv(history_path, index=False)

    if save_predictions:
        np.savez_compressed(
            prediction_path,
            train_pred=evaluation["Train"]["predictions"],
            train_true=evaluation["Train"]["ground_truth"],
            test_pred=evaluation["Test"]["predictions"],
            test_true=evaluation["Test"]["ground_truth"],
        )

    peak_vram_gb = (
        float(torch.cuda.max_memory_allocated() / (1024 ** 3))
        if DEVICE.type == "cuda" else None
    )
    record = {
        "status": "complete",
        "config": config,
        "dataset_status": str(bundle["status"]),
        "train_samples": int(len(x_train)),
        "test_samples": int(len(x_test)),
        "train_mse": float(evaluation["Train"]["MSE"]),
        "train_mae": float(evaluation["Train"]["MAE"]),
        "test_mse": float(evaluation["Test"]["MSE"]),
        "test_mae": float(evaluation["Test"]["MAE"]),
        "first_training_loss": float(history["loss"][0]),
        "last_training_loss": float(history["loss"][-1]),
        "last_weighted_lyap": float(history["lyap_weighted"][-1]),
        "training_seconds_this_session": float(training_seconds_this_session),
        "evaluation_seconds": float(evaluation_seconds),
        "peak_pytorch_vram_gb": peak_vram_gb,
        "checkpoint": str(checkpoint_path),
    }

    # Save metrics LAST, so a completed metrics file means every artifact exists.
    temp_path = metrics_path.with_name(metrics_path.name + ".tmp")
    with temp_path.open("w", encoding="utf-8") as stream:
        json.dump(record, stream, indent=2, ensure_ascii=False, allow_nan=False)
    temp_path.replace(metrics_path)

    print(
        f"[DONE] {dataset_name} | {model_name} | P={patch_len}, H={horizon} | "
        f"Test MSE={record['test_mse']:.6f} | MAE={record['test_mae']:.6f} | "
        f"session={training_seconds_this_session:.1f}s"
    )

    del model, optimizer, evaluation
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record

## 8. Ablation Grid Planning and Execution

Phần này xây dựng danh sách các cấu hình thực nghiệm dựa trên tích Descartes của dataset, model, patch length và forecast horizon.

### Experiment grid

- Datasets: 4
- Models: 6
- Patch lengths: 7
- Forecast horizons: 8
- Epochs: 4000
- Seed: 7

Tổng cộng có 1344 cấu hình thực nghiệm.

Hàm `build_ablation_grid()` xây dựng bảng cấu hình, kiểm tra số lượng sliding windows và phát hiện các cấu hình trùng lặp.

Grid được lưu thành `ablation_grid_plan.csv` để phục vụ quản lý thực nghiệm.

### Execution controls

Các tùy chọn `RUN_FULL_GRID`, `ONLY_DATASETS`, `ONLY_MODELS` và `MAX_RUNS` dùng để kiểm soát phạm vi thực thi.

Mặc định `RUN_FULL_GRID=False`, do đó notebook chỉ lập kế hoạch mà không bắt đầu huấn luyện.

Notebook này giữ lại logic grid để tham khảo và tái lập quy trình. Các lượt huấn luyện thực tế cũng có thể được thực hiện bằng script `train_ablation_v2.py`.

In [12]:
from itertools import product


def build_ablation_grid():
    rows = []
    for dataset_name, patch_len, horizon, model_name in product(
        ABLATION_DATASETS.keys(), PATCH_LENGTHS, HORIZONS, MODEL_VARIANTS
    ):
        bundle = ABLATION_DATASETS[dataset_name]
        n_train = len(bundle["train"]) - patch_len - horizon
        n_test = len(bundle["test"]) - patch_len - horizon
        if min(n_train, n_test) <= 0:
            raise ValueError(f"Insufficient windows: {dataset_name} P={patch_len} H={horizon}")
        rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "P": int(patch_len),
            "H": int(horizon),
            "TrainWindows": int(n_train),
            "TestWindows": int(n_test),
            "Epochs": int(EPOCHS),
            "Seed": int(SEED),
        })
    result = pd.DataFrame(rows)
    expected = (
        len(ABLATION_DATASETS) * len(PATCH_LENGTHS)
        * len(HORIZONS) * len(MODEL_VARIANTS)
    )
    assert len(result) == expected == 1344
    assert not result.duplicated(["Dataset", "Model", "P", "H"]).any()
    return result


ABLATION_GRID = build_ablation_grid()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_GRID.to_csv(RESULTS_DIR / "ablation_grid_plan.csv", index=False)

print("=== ABLATION GRID PLAN ===")
print("Datasets:", ABLATION_GRID["Dataset"].nunique())
print("Models:", ABLATION_GRID["Model"].nunique())
print("Patch lengths:", ABLATION_GRID["P"].nunique())
print("Horizons:", ABLATION_GRID["H"].nunique())
print("Total configurations:", len(ABLATION_GRID))
display(ABLATION_GRID.head(12))

# IMPORTANT: leave False on your laptop.
# Switch to True only when you are ready to execute the complete grid.
RUN_FULL_GRID = False

# Optional filters on the rented GPU: None means ALL.
ONLY_DATASETS = None      # e.g. ["CMIP6 Wind"]
ONLY_MODELS = None        # e.g. ["KoopPatchTST_Full"]
MAX_RUNS = None           # e.g. 2 for a short real trial


selected_grid = ABLATION_GRID.copy()
if ONLY_DATASETS is not None:
    selected_grid = selected_grid[selected_grid["Dataset"].isin(ONLY_DATASETS)]
if ONLY_MODELS is not None:
    selected_grid = selected_grid[selected_grid["Model"].isin(ONLY_MODELS)]
if MAX_RUNS is not None:
    selected_grid = selected_grid.head(MAX_RUNS)

print("Selected configurations:", len(selected_grid))

if not RUN_FULL_GRID:
    print("DRY RUN ONLY — no 4000-epoch training started.")
    print("Cell 8 completed successfully — grid is ready for the rented GPU.")
else:
    if len(selected_grid) == 0:
        raise ValueError("No configurations selected.")

    for job_index, row in enumerate(selected_grid.itertuples(index=False), start=1):
        print(
            f"\n[{job_index}/{len(selected_grid)}] "
            f"{row.Dataset} | {row.Model} | P={row.P}, H={row.H}"
        )
        # An existing completed metrics file is skipped.
        # An existing checkpoint without metrics is resumed.
        # Exceptions stop the grid so that errors are investigated.
        run_one_ablation(
            dataset_name=row.Dataset,
            model_name=row.Model,
            patch_len=int(row.P),
            horizon=int(row.H),
            epochs=EPOCHS,
            seed=SEED,
            checkpoint_every=500,
            save_predictions=False,
            verbose=False,
        )

    print("\nSelected ablation grid finished.")
    print("Run Cell 9 to aggregate completed metrics.")


=== ABLATION GRID PLAN ===
Datasets: 4
Models: 6
Patch lengths: 7
Horizons: 8
Total configurations: 1344


,Dataset,Model,P,H,TrainWindows,TestWindows,Epochs,Seed
0,Electricity,DLinear,70,2,2728,628,4000,7
1,Electricity,LSTM,70,2,2728,628,4000,7
2,Electricity,PatchTST,70,2,2728,628,4000,7
3,Electricity,PatchTST+LinearLatent,70,2,2728,628,4000,7
4,Electricity,KoopPatchTST_NoLyap,70,2,2728,628,4000,7
5,Electricity,KoopPatchTST_Full,70,2,2728,628,4000,7
6,Electricity,DLinear,70,4,2726,626,4000,7
7,Electricity,LSTM,70,4,2726,626,4000,7
8,Electricity,PatchTST,70,4,2726,626,4000,7
9,Electricity,PatchTST+LinearLatent,70,4,2726,626,4000,7


Selected configurations: 1344
DRY RUN ONLY — no 4000-epoch training started.
Cell 8 completed successfully — grid is ready for the rented GPU.



## 9. Ablation Experiment Execution

Phần này cung cấp giao diện điều khiển thực nghiệm trực tiếp từ Jupyter Notebook, thay cho việc gọi các script Python bên ngoài.

### Execution controls

- `RUN_ABLATION_TRAINING`: Cho phép bắt đầu huấn luyện.
- `ONLY_DATASETS`: Lựa chọn dataset.
- `ONLY_MODELS`: Lựa chọn mô hình.
- `ONLY_PATCHES`: Lựa chọn patch length.
- `ONLY_HORIZONS`: Lựa chọn forecast horizon.
- `MAX_RUNS`: Giới hạn số cấu hình mới được thực hiện.

### Existing results protection

Trước khi huấn luyện, notebook đối chiếu cấu hình với bảng kết quả đã tổng hợp trong `ablation_merged_1344/metrics.csv`.

Các cấu hình đã có đầy đủ Train/Test metrics được bỏ qua.

Notebook không tự động khởi động lại hoặc ghi đè checkpoint do các script cũ tạo ra.

### Execution

Mặc định `RUN_ABLATION_TRAINING=False` nên chỉ kiểm tra kế hoạch và không thực hiện huấn luyện.

Khi cần chạy cấu hình mới, người thực hiện phải chủ động bật chế độ training.

In [13]:
# 1. Execution controls
RUN_ABLATION_TRAINING = False

ONLY_DATASETS = None
ONLY_MODELS = None
ONLY_PATCHES = None
ONLY_HORIZONS = None

MAX_RUNS = None

# 2. Load completed experiment registry (read-only)
MERGED_METRICS = (
    RESULTS_DIR
    / "ablation_merged_1344"
    / "metrics.csv"
)

if not MERGED_METRICS.is_file():
    raise FileNotFoundError(
        f"Completed-run registry not found: {MERGED_METRICS}"
    )

completed_df = pd.read_csv(MERGED_METRICS)

keys = [
    "Dataset", "Model", "PatchLen",
    "Horizon", "Epochs", "Seed"
]

required_columns = keys + ["Set"]

if not set(required_columns).issubset(completed_df.columns):
    raise ValueError("Invalid merged metrics schema")

if completed_df.duplicated(keys + ["Set"]).any():
    raise ValueError("Duplicate completed metrics found")

completed_runs = set()

for config, group in completed_df.groupby(keys):
    if {"Train", "Test"}.issubset(set(group["Set"])):
        completed_runs.add(tuple(config))

# 3. Select configurations
selected_grid = ABLATION_GRID.copy()

filters = {
    "Dataset": ONLY_DATASETS,
    "Model": ONLY_MODELS,
    "P": ONLY_PATCHES,
    "H": ONLY_HORIZONS
}

for column, allowed in filters.items():
    if allowed is not None:
        selected_grid = selected_grid[
            selected_grid[column].isin(allowed)
        ]

# 4. Identify unfinished configurations
pending = []

for row in selected_grid.itertuples(index=False):
    key = (
        row.Dataset,
        row.Model,
        int(row.P),
        int(row.H),
        int(row.Epochs),
        int(row.Seed)
    )

    if key not in completed_runs:
        pending.append(row)

if MAX_RUNS is not None:
    if MAX_RUNS <= 0:
        raise ValueError("MAX_RUNS must be positive")
    pending = pending[:MAX_RUNS]

print("=== ABLATION EXECUTION PLAN ===")
print("Selected:", len(selected_grid))
print("Already completed:",
      len(selected_grid) -
      sum(
          (
              row.Dataset, row.Model,
              int(row.P), int(row.H),
              int(row.Epochs), int(row.Seed)
          ) not in completed_runs
          for row in selected_grid.itertuples(index=False)
      ))
print("Pending runs selected:", len(pending))

# 5. Execute only after explicit opt-in
if not RUN_ABLATION_TRAINING:
    print("DRY RUN — no training started.")

else:
    for index, row in enumerate(pending, 1):
        # Legacy CLI checkpoint paths are not compatible
        # with the notebook checkpoint schema.
        legacy_dir = (
            RESULTS_DIR / "ablation_runs"
            / _safe_slug(row.Dataset)
            / f"p{row.P}_h{row.H}"
            / _safe_slug(row.Model)
            / f"e{EPOCHS}_seed{SEED}"
        )

        merged_legacy_dir = (
            RESULTS_DIR / "ablation_merged_1344"
            / "ablation_runs"
            / _safe_slug(row.Dataset)
            / f"p{row.P}_h{row.H}"
            / _safe_slug(row.Model)
            / f"e{EPOCHS}_seed{SEED}"
        )

        for directory in (legacy_dir, merged_legacy_dir):
            if (
                (directory / "latest.pt").exists()
                or (directory / "metrics.json").exists()
            ):
                raise RuntimeError(
                    "Existing legacy artifacts require "
                    f"compatibility review: {directory}"
                )

        print(
            f"[{index}/{len(pending)}] "
            f"{row.Dataset} | {row.Model} "
            f"P={row.P} H={row.H}"
        )

        run_one_ablation(
            dataset_name=row.Dataset,
            model_name=row.Model,
            patch_len=int(row.P),
            horizon=int(row.H),
            epochs=EPOCHS,
            seed=SEED,
            checkpoint_every=500,
            save_predictions=False,
            verbose=False,
            force_restart=False
        )

    print("Selected configurations finished.")


=== ABLATION EXECUTION PLAN ===
Selected: 1344
Already completed: 1344
Pending runs selected: 0
DRY RUN — no training started.
